# Comparison of PME.csv and PME2.csv
This notebook analyzes the differences between the two datasets.

In [41]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('/Volumes/shared/HAR_WG/WG/UKSEA_VAXHUB/github_repo/Systematic_Review_Dengue_Forecasting/data/PME.csv')
df2 = pd.read_csv('/Volumes/shared/HAR_WG/WG/UKSEA_VAXHUB/github_repo/Systematic_Review_Dengue_Forecasting/data/PME2.csv')

print(f'PME.csv shape: {df1.shape}')
print(f'PME2.csv shape: {df2.shape}')

PME.csv shape: (1991, 28)
PME2.csv shape: (1991, 27)


## 1. Column Differences

In [42]:
cols1 = set(df1.columns)
cols2 = set(df2.columns)


print('Couluns in PME:', df1.columns)
print('\nCouluns in PME2:', df2.columns)

print("\n")

print('Columns in PME but not in PME2:', cols1 - cols2)
print('Columns in PME2 but not in PME:', cols2 - cols1)

Couluns in PME: Index(['Key', 'Year', 'Identifier', 'Model', 'Model2', 'Category',
       'Validation', 'Training (%)', 'Test (%)', 'Time Horizon',
       'Time Horizon (Months)', 'Rank within study', 'RMSE', 'nRMSE',
       'R-Squared', 'R', 'MAE', 'MAPE', 'MASE', 'CORR', 'ME', 'SMAPE', 'MSE',
       'Lambda', 'CVE', 'Pearson's correlation coefficient (ρ)',
       'Note/Input Variables', 'EXCLUDE'],
      dtype='object')

Couluns in PME2: Index(['Key', 'Year', 'Identifier', 'Model', 'Model2', 'Category',
       'Validation', 'Training (%)', 'Test (%)', 'Time Horizon',
       'Rank within study', 'RMSE', 'nRMSE', 'R-Squared', 'R', 'MAE', 'MAPE',
       'MASE', 'CORR', 'ME', 'SMAPE', 'MSE', 'Lambda', 'CVE',
       'Pearson's correlation coefficient (ρ)', 'Note/Input Variables',
       'EXCLUDE'],
      dtype='object')


Columns in PME but not in PME2: {'Time Horizon (Months)'}
Columns in PME2 but not in PME: set()


## 2. Row/Identifier Comparison
Comparing based on the 'Key' column.

In [43]:
keys1 = set(df1['Key'].unique())
keys2 = set(df2['Key'].unique())

print(f'Keys unique to PME: {len(keys1 - keys2)}')
print(f'Keys unique to PME2: {len(keys2 - keys1)}')
print(f'Shared Keys: {len(keys1 & keys2)}')

Keys unique to PME: 0
Keys unique to PME2: 0
Shared Keys: 59


In [ ]:
# Standardize column types for comparison
# 'Rank within study' is numeric in one file and text in the other
df1['Rank within study'] = df1['Rank within study'].astype(str)
df2['Rank within study'] = df2['Rank within study'].astype(str)

# Create a Composite ID to identify unique rows across both files
# We use Key, Identifier, Year, and Rank to define a unique record
def create_id(df):
    return (
        df['Key'].astype(str) + "_" + 
        df['Identifier'].astype(str) + "_" + 
        df['Year'].astype(str) + "_" + 
        df['Rank within study'].astype(str)
    )

df1['composite_id'] = create_id(df1)
df2['composite_id'] = create_id(df2)

# Find rows in PME2 that do not exist in PME
new_rows_mask = ~df2['composite_id'].isin(df1['composite_id'])
df_difference = df2[new_rows_mask].copy()

# Clean up and display
# Remove the helper column before showing/saving
df_difference = df_difference.drop(columns=['composite_id'])

print(f"Found {len(df_difference)} new rows in PME2.")

# Display the first few rows to see the updates
print(df_difference[['Key', 'Identifier', 'Year', 'Model', 'RMSE']].head())

Found 0 new rows in PME2.
Empty DataFrame
Columns: [Key, Identifier, Year, Model, RMSE]
Index: []


## 3. Data Value Comparison
Checking for value discrepancies in shared columns for the same 'Key'.

In [45]:
common_cols = sorted(list(cols1.intersection(cols2)))
# We pick a subset of columns to compare for simplicity
subset_cols = ['Key', 'Year', 'Identifier', 'RMSE', 'MAE']
comparison = pd.merge(df1, df2, on=['Key', 'Identifier', 'Year'], suffixes=('_v1', '_v2'))
print(f'Number of matching rows by Key/ID/Year: {len(comparison)}')

# Check for RMSE differences
comparison['RMSE_v1'] = pd.to_numeric(comparison['RMSE_v1'], errors='coerce')
comparison['RMSE_v2'] = pd.to_numeric(comparison['RMSE_v2'], errors='coerce')
diff_mask = (comparison['RMSE_v1'] != comparison['RMSE_v2']) & comparison['RMSE_v1'].notnull()
print(f'Rows with differing RMSE: {diff_mask.sum()}')
if diff_mask.sum() > 0:
    display(comparison[diff_mask][['Key', 'RMSE_v1', 'RMSE_v2']].head())

Number of matching rows by Key/ID/Year: 216509
Rows with differing RMSE: 171721


,Key,RMSE_v1,RMSE_v2
26,Z7R5IGNY,0.5487,0.5597
27,Z7R5IGNY,0.5487,119.3177
28,Z7R5IGNY,0.5597,0.5487
30,Z7R5IGNY,0.5597,119.3177
31,Z7R5IGNY,119.3177,0.5487
